In [1]:
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import json
import gseapy as gp

In [2]:
DISEASE = input("Disease: ")
DISEASE_FOLDER = f"../output/{DISEASE}/"
RESULT_FOLDER = DISEASE_FOLDER + "leiden_results"

# Loading

In [3]:
def load(d):
    with open(f"../output/{d}/leiden_results/result_communities_selected.pkl", "rb") as f:
        communities = pickle.load(f)
    with open(f"../output/{d}/leiden_results/result_communities_HGNC_selected.pkl", "rb") as f:
        communities_HGNC = pickle.load(f)
    # with open(f"../output/{d}/leiden_results/result_graph.pkl", "rb") as f:
    #     graph = pickle.load(f)    
    with open(f"../output/{d}/gene_to_index_distinct.json", "r") as file:
        gene_to_index_distinct = json.load(file)
        
    return communities,communities_HGNC,gene_to_index_distinct

In [4]:
communities_selected,communities_HGNC_selected,gene_to_index_distinct = load(DISEASE)

# index to HGNC

In [5]:
index_to_gene_distinct = {v: u for (u,v) in gene_to_index_distinct.items()}

In [6]:
hgnc = pd.read_csv("../../Data/hgnc_complete_set.txt", sep="\t", dtype=str)
ncbi_to_hgnc_dict = dict(
    zip(
        hgnc["entrez_id"].dropna(),
        hgnc.loc[hgnc["entrez_id"].notna(), "symbol"]
    )
)

# Select top genes

In [7]:
# comm_idx = 0

In [8]:
def zscore(values):
    arr = np.asarray(values, dtype=float)
    if arr.size == 0:
        return arr

    mean = arr.mean()
    std = arr.std(ddof=0)

    if std == 0 or np.isnan(std):
        # no variation: all z-scores = 0
        return np.zeros_like(arr)

    return (arr - mean) / std

def community_central_genes_ranked(G, community_nodes, weight="weight"):
    C = set(community_nodes)
    H = G.subgraph(C).copy()                       # induced subgraph
    # within-community (weighted) degree
    k = {u: H.degree(u, weight=weight) for u in H}
    ks = np.array(list(k.values()), dtype=float)
    zscore_list = zscore(ks)
    Z = dict(zip(H,zscore_list))        # within-module degree z-score
    
    return Z

In [9]:
index_to_gene_distinct

{10: '1',
 11: '2',
 12: '3',
 13: '9',
 14: '10',
 15: '12',
 16: '13',
 17: '14',
 18: '15',
 19: '16',
 20: '18',
 21: '19',
 22: '20',
 23: '21',
 24: '22',
 25: '23',
 26: '24',
 27: '25',
 28: '26',
 29: '27',
 30: '28',
 31: '29',
 32: '30',
 33: '31',
 34: '32',
 35: '33',
 36: '34',
 37: '35',
 38: '36',
 39: '37',
 40: '38',
 41: '39',
 42: '40',
 43: '41',
 44: '43',
 45: '47',
 46: '48',
 47: '49',
 48: '50',
 49: '51',
 50: '52',
 51: '53',
 52: '54',
 53: '55',
 54: '56',
 55: '58',
 56: '59',
 57: '60',
 58: '69',
 59: '70',
 60: '71',
 61: '72',
 62: '81',
 63: '83',
 64: '86',
 65: '87',
 66: '88',
 67: '89',
 68: '90',
 69: '91',
 70: '92',
 71: '93',
 72: '94',
 73: '95',
 74: '97',
 75: '98',
 76: '100',
 77: '101',
 78: '102',
 79: '103',
 80: '104',
 81: '105',
 82: '107',
 83: '108',
 84: '109',
 85: '111',
 86: '112',
 87: '113',
 88: '114',
 89: '115',
 90: '116',
 91: '117',
 92: '118',
 93: '119',
 94: '120',
 95: '123',
 96: '124',
 97: '125',
 98: '126',
 9

In [10]:
# zscore_dict = community_central_genes_ranked(graph,communities[comm_idx])

In [11]:
# zscore_dict = dict(sorted(zscore_dict.items(), key=lambda x: x[1], reverse=True))

In [12]:
# zscore_dict

In [13]:
# top_genes = [u for u,v in zscore_dict.items()][:3]

In [14]:
# top_genes

# Community deepdive

In [15]:
comm_idx = 0

In [16]:
comm_HGNC = communities_HGNC_selected[comm_idx]

In [17]:
len(comm_HGNC)

1369

In [18]:
important_terms = pd.read_csv(DISEASE_FOLDER + "important_terms.csv")

In [19]:
go_df_filtered = important_terms[important_terms["Community Index"] == comm_idx]

In [20]:
go_df_filtered = go_df_filtered.sort_values(by = ["Adjusted P-value"], ascending = [True])

In [21]:
go_df_filtered

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,GO_ID,Slim_IDs,Overlap (value),KEGG_ID
213,0,1369,Immune System R-HSA-168256,285/1943,1.026963e-34,['Immune System'],Reactome_2022,8.853125e-38,0.0,0.0,2.691471,229.629455,CYFIP2;IFITM3;CNTFR;CYFIP1;IFITM1;IL1RN;IFITM2...,NaN,NaN,0.146680,NaN
109,0,1369,Regulation Of Apoptotic Process (GO:0042981),126/705,2.939267e-20,['biological regulation'],GO_Biological_Process_2023,7.108263e-24,0.0,0.0,3.160430,168.453399,FOXA1;PHB1;TFRC;PDCD5;FAIM2;IFIT3;IFIT2;IGF1R;...,GO:0042981,{'GO:0065007'},0.178723,NaN
221,0,1369,Signal Transduction R-HSA-162582,289/2465,7.961780e-19,['Signal Transduction'],Reactome_2022,1.372721e-21,0.0,0.0,2.023546,97.206076,CYFIP2;CYFIP1;PHB1;TRIO;TFRC;HHIP;ITSN1;STMN2;...,NaN,NaN,0.117241,NaN
206,0,1369,Cytokine Signaling In Immune System R-HSA-1280215,120/702,1.650074e-18,['Immune System'],Reactome_2022,4.267434e-21,0.0,0.0,2.979538,139.750101,IFITM3;CNTFR;IFITM1;IL1RN;IFITM2;IFIT5;F13A1;U...,NaN,NaN,0.170940,NaN
67,0,1369,Positive Regulation Of Intracellular Signal Tr...,100/525,5.802099e-18,['biological regulation'],GO_Biological_Process_2023,3.557771e-21,0.0,0.0,3.375701,158.945402,PHB1;TFRC;NCF1;TRAF3IP2;IFIT5;IFI35;IGF1R;TBK1...,GO:1902533,{'GO:0065007'},0.190476,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,0,1369,Negative Regulation Of Angiogenesis (GO:0016525),17/86,9.234840e-04,['biological regulation'],GO_Biological_Process_2023,6.253338e-05,0.0,0.0,3.382579,32.742719,SLC12A2;FOXC1;SEMA6A;SERPINF1;PTPRM;NRG1;VASH1...,GO:0016525,{'GO:0065007'},0.197674,NaN
391,0,1369,Regulation Of Actin Filament-Based Process (GO...,15/70,9.315534e-04,['biological regulation'],GO_Biological_Process_2023,6.330508e-05,0.0,0.0,3.741641,36.172482,RALA;RHOG;ARHGAP18;NEDD9;NRG1;FZD10;F11R;IQGAP...,GO:0032970,{'GO:0065007'},0.214286,NaN
126,0,1369,Collagen Degradation R-HSA-1442490,11/40,9.784876e-04,['Extracellular matrix organization'],Reactome_2022,5.398552e-05,0.0,0.0,5.195826,51.058310,MMP12;MMP14;MMP7;MMP13;TMPRSS6;MMP1;CTSK;MMP2;...,NaN,NaN,0.275000,NaN
277,0,1369,Sterol Biosynthetic Process (GO:0016126),9/28,9.850395e-04,['cellular process'],GO_Biological_Process_2023,6.741625e-05,0.0,0.0,6.482508,62.262053,MVK;HMGCS1;INSIG2;INSIG1;HMGCR;DHCR7;LSS;TM7SF...,GO:0016126,{'GO:0009987'},0.321429,NaN


In [22]:
go_df_filtered[go_df_filtered["Gene_set"] == "Reactome_2022"]

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,GO_ID,Slim_IDs,Overlap (value),KEGG_ID
213,0,1369,Immune System R-HSA-168256,285/1943,1.026963e-34,['Immune System'],Reactome_2022,8.853125e-38,0.0,0.0,2.691471,229.629455,CYFIP2;IFITM3;CNTFR;CYFIP1;IFITM1;IL1RN;IFITM2...,NaN,NaN,0.146680,NaN
221,0,1369,Signal Transduction R-HSA-162582,289/2465,7.961780e-19,['Signal Transduction'],Reactome_2022,1.372721e-21,0.0,0.0,2.023546,97.206076,CYFIP2;CYFIP1;PHB1;TRIO;TFRC;HHIP;ITSN1;STMN2;...,NaN,NaN,0.117241,NaN
206,0,1369,Cytokine Signaling In Immune System R-HSA-1280215,120/702,1.650074e-18,['Immune System'],Reactome_2022,4.267434e-21,0.0,0.0,2.979538,139.750101,IFITM3;CNTFR;IFITM1;IL1RN;IFITM2;IFIT5;F13A1;U...,NaN,NaN,0.170940,NaN
217,0,1369,Innate Immune System R-HSA-168249,149/1035,3.632325e-16,['Immune System'],Reactome_2022,1.252526e-18,0.0,0.0,2.446069,100.830317,CYFIP2;OTUD5;CYFIP1;NCF1;DEFB1;UBE2L6;LGALS3;R...,NaN,NaN,0.143961,NaN
131,0,1369,Interferon Signaling R-HSA-913531,51/200,2.351027e-14,['Immune System'],Reactome_2022,1.013374e-16,0.0,0.0,4.799737,176.765090,IFITM3;IFITM1;IFITM2;IFIT5;UBE2L6;IFI35;IFIT1;...,NaN,NaN,0.255000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116,0,1369,Defective B3GAT3 Causes JDSSDHD R-HSA-3560801,8/20,5.555088e-04,['Disease'],Reactome_2022,2.825433e-05,0.0,0.0,9.120255,95.527955,SDC4;GPC3;SDC1;CSPG4;GPC4;AGRN;GPC6;DCN,NaN,NaN,0.400000,NaN
210,0,1369,TCF Dependent Signaling In Response To WNT R-H...,30/198,6.510225e-04,['Signal Transduction'],Reactome_2022,3.367358e-05,0.0,0.0,2.462259,25.358302,USP34;WNT8A;LRP6;ZNRF3;TNKS2;RSPO2;DVL2;RSPO3;...,NaN,NaN,0.151515,NaN
119,0,1369,GAG Synthesis Requires Tetrasaccharide Linker ...,9/26,6.570665e-04,['Metabolism'],Reactome_2022,3.455264e-05,0.0,0.0,7.245934,74.437676,SDC4;GPC3;XYLT1;SDC1;CSPG4;GPC4;AGRN;GPC6;DCN,NaN,NaN,0.346154,NaN
196,0,1369,Axon Guidance R-HSA-422475,60/519,8.745334e-04,['Developmental Biology'],Reactome_2022,4.674230e-05,0.0,0.0,1.814687,18.093988,NRP1;TRIO;RPL31;SH3KBP1;ITSN1;CASC3;VLDLR;LAMC...,NaN,NaN,0.115607,NaN


In [23]:
print(go_df_filtered[["Term","Adjusted P-value","Overlap"]].to_latex(longtable=True,index=False,caption=f"Enriched terms for {DISEASE} Community {comm_idx}",label=f"appendix:{DISEASE}_{comm_idx}_term", float_format="%.2e",column_format="p{8cm}cc",))

\begin{longtable}{p{8cm}cc}
\caption{Enriched terms for BIPOLAR Community 0} \label{appendix:BIPOLAR_0_term} \\
\toprule
Term & Adjusted P-value & Overlap \\
\midrule
\endfirsthead
\caption[]{Enriched terms for BIPOLAR Community 0} \\
\toprule
Term & Adjusted P-value & Overlap \\
\midrule
\endhead
\midrule
\multicolumn{3}{r}{Continued on next page} \\
\midrule
\endfoot
\bottomrule
\endlastfoot
Immune System R-HSA-168256 & 1.03e-34 & 285/1943 \\
Regulation Of Apoptotic Process (GO:0042981) & 2.94e-20 & 126/705 \\
Signal Transduction R-HSA-162582 & 7.96e-19 & 289/2465 \\
Cytokine Signaling In Immune System R-HSA-1280215 & 1.65e-18 & 120/702 \\
Positive Regulation Of Intracellular Signal Transduction (GO:1902533) & 5.80e-18 & 100/525 \\
Regulation Of Cell Population Proliferation (GO:0042127) & 5.80e-18 & 127/766 \\
Defense Response To Virus (GO:0051607) & 3.85e-17 & 54/189 \\
Wnt Signaling Pathway (GO:0016055) & 3.85e-17 & 34/76 \\
Negative Regulation Of Cellular Process (GO:0048523) & 4

In [24]:
list(go_df_filtered["Term"])

['Immune System R-HSA-168256',
 'Regulation Of Apoptotic Process (GO:0042981)',
 'Signal Transduction R-HSA-162582',
 'Cytokine Signaling In Immune System R-HSA-1280215',
 'Positive Regulation Of Intracellular Signal Transduction (GO:1902533)',
 'Regulation Of Cell Population Proliferation (GO:0042127)',
 'Defense Response To Virus (GO:0051607)',
 'Wnt Signaling Pathway (GO:0016055)',
 'Negative Regulation Of Cellular Process (GO:0048523)',
 'Innate Immune System R-HSA-168249',
 'Defense Response To Symbiont (GO:0140546)',
 'Collagen-Containing Extracellular Matrix (GO:0062023)',
 'Wnt signaling pathway',
 'Frizzled Binding (GO:0005109)',
 'Regulation Of I-kappaB kinase/NF-kappaB Signaling (GO:0043122)',
 'Negative Regulation Of Cell Population Proliferation (GO:0008285)',
 'Positive Regulation Of DNA-templated Transcription (GO:0045893)',
 'Proteoglycans in cancer',
 'Regulation Of Cell Migration (GO:0030334)',
 'Canonical Wnt Signaling Pathway (GO:0060070)',
 'Basal cell carcinoma',


In [25]:
list(go_df_filtered["Category"].unique())

["['Immune System']",
 "['biological regulation']",
 "['Signal Transduction']",
 "['response to stimulus', 'biological process involved in interspecies interaction between organisms']",
 "['biological regulation', 'cellular process']",
 '[]',
 "['Signal transduction']",
 "['binding']",
 "['Cancer: overview']",
 "['Cancer: specific types']",
 "['cellular anatomical structure']",
 "['Cellular community - eukaryotes']",
 "['Hemostasis']",
 "['localization', 'cellular process']",
 "['developmental process', 'cellular process']",
 "['Extracellular matrix organization']",
 "['response to stimulus']",
 "['cellular process']",
 "['catalytic activity']",
 "['developmental process']",
 "['Endocrine system']",
 "['response to stimulus', 'cellular process']",
 "['molecular transducer activity']",
 "['Developmental Biology']",
 "['response to stimulus', 'immune system process', 'biological process involved in interspecies interaction between organisms']",
 "['Cell motility']",
 "['Infectious diseas